# [실습] LangChain과 다양한 툴 연동하기  

LangChain의 Tool은 OpenAI의 Tool과 유사합니다.

Tool은 LLM이 답변을 출력하기 위해 활용할 수 있는 다양한 수단을 의미합니다.   
LLM은 Tool에 입력을 전달하고, 답변을 파싱하는 작업을 수행합니다.


In [ ]:
!pip install openai matplotlib langchain langchain-openai langchain-community langchain-tavily langgraph -q

In [ ]:
import os
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI

load_dotenv(override=True)

llm = ChatOpenAI(model = 'gpt-5.2', reasoning_effort='low')

Tavily Search (http://app.tavily.com/)

Tavily Search는 1달에 1,000회 검색을 지원하는 검색 API입니다.    
구글 계정을 통해 회원가입할 수 있습니다.

In [ ]:
from langchain_tavily import TavilySearch

# TAVILY : 웹 검색을 수행하는 툴
# TAVILY_API_KEY https://app.tavily.com/home

In [ ]:
load_dotenv(override=True)

tavily_search = TavilySearch(max_results=3)

tavily_search.invoke("Gemma 4")

다양한 툴을 추가해 보겠습니다.

In [ ]:
from langchain_core.tools import tool

@tool
def calculator(expression: str) -> str:
    """수학 계산식을 입력받아 계산 결과를 반환합니다.
    사칙연산, 거듭제곱(**) 등 Python 수식을 지원합니다.

    Args:
        expression: 계산할 수식 (예: '3 + 5 * 2', '2 ** 10')
    """
    try:
        allowed_chars = set('0123456789+-*/.()**% ')
        if not all(c in allowed_chars for c in expression):
            return f'허용되지 않는 문자가 포함되어 있습니다: {expression}'
        # 주의: 교육용 코드입니다. 프로덕션에서는 ast.literal_eval 또는 numexpr을 사용하세요.
        result = eval(expression)
        return str(result)
    except Exception as e:
        return f'계산 오류: {str(e)}'

print(f"calculator('3 + 5 * 2') = {calculator.invoke({'expression': '3 + 5 * 2'})}")

@tool
def current_date() -> str:
    "현재 날짜를 %y-%m-%d 형식으로 반환합니다."
    from datetime import datetime
    return f'현재 날짜는 {datetime.now().strftime("%Y-%m-%d")} 입니다!'


print(current_date.invoke({}))

위에서 만든 툴을 리스트로 묶고, LLM에 binding합니다.

In [ ]:
tools = [tavily_search, calculator, current_date]

In [ ]:
llm_with_tools = llm.bind_tools(tools)
llm_with_tools

llm_with_tools는 llm에 tool이 결합된 형태지만, llm의 구조입니다.   
invoke를 통해 툴을 사용하도록 유도해 봅시다.

In [ ]:
# 툴 실행을 하지 않음
llm_with_tools.invoke("안녕?")

In [ ]:
llm_with_tools.invoke("29392 * 23919는 뭐야?")

In [ ]:
# 툴 실행을 하지 않을 확률이 높음
llm_with_tools.invoke("양자 컴퓨터의 정의가 뭐야?")

In [ ]:
llm_with_tools.invoke("양자 컴퓨터 최신 소식 있어?")

content를 생성하는 대신에, tool_calls가 도출된 것을 볼 수 있습니다.

In [ ]:
llm_with_tools.invoke("29392 * 23919는 뭐야?").tool_calls

In [ ]:
llm_with_tools.invoke("오늘 운동을 빠질까요?").tool_calls
# No Tool Call

Tool을 실행할 때, tool_call 정보를 보내면 ToolMessage 형식의 결과가 생성됩니다.   

이후, 이 내용을 메시지에 넣어서 전달하면 됩니다.

In [ ]:
calculator.invoke(llm_with_tools.invoke("29392 * 23919는 뭐야?").tool_calls[0])

In [ ]:
# Message 전달하기 예시
from langchain.messages import HumanMessage, AIMessage, ToolMessage

question = '29392 * 23919가 뭐야?'
tool_call_msg = llm_with_tools.invoke(question)
tool_msg = calculator.invoke(tool_call_msg.tool_calls[0])

msgs = [HumanMessage(question),
       tool_call_msg,
       tool_msg]

llm_with_tools.invoke(msgs)

### Agent(에이전트)

에이전트는 위에서 구성한 Tool Calling 과정을 반복하여 작업을 완료하는 구조입니다.   


llm 실행 --> Tool 요청 --> Tool 실행 결과 전달 --> llm 실행 --> 의 과정을 반복합니다.   

In [ ]:
tools = [calculator, tavily_search, current_date]

In [ ]:
from langchain.agents import create_agent

system_prompt='''툴을 사용하기 전, 툴 사용 계획에 대해 설명하고 중간 단계를 매번 간략하게 요약하세요..'''

agent = create_agent(llm, tools=tools, system_prompt= system_prompt)

agent


In [ ]:
question = "오늘 날짜를 확인해줘."

response = agent.invoke({'messages':[HumanMessage(question)]})

response['messages'][-1].text

In [ ]:
response

In [ ]:
question = "Gemma 4 모델의 장단점을 조사해서 마크다운 표로 작성해줘."

response = agent.invoke({'messages':[HumanMessage(question)]})

response['messages'][-1].text

In [ ]:
# 작동 과정
response

스트리밍을 통해, 단계별 작업 과정도 확인합니다.

In [ ]:
def stream_agent(agent, input_msgs, config=None):
    """에이전트 실행을 스트리밍하며 각 스텝을 출력합니다."""
    def _format_arg(v):
        s = repr(v) if not isinstance(v, str) else v
        return s if len(s) <= 200 else s[:197] + "..."

    last_msg = None
    for chunk in agent.stream(input_msgs, config=config, stream_mode="updates"):
        for step, data in chunk.items():
            msg = data['messages'][-1]
            content = msg.text if hasattr(msg, 'text') else str(msg.content)
            print(f"[{step}]")
            if len(content) > 1000:
                print(f"  content: {content[:800]}...(중략)")
            else:
                print(f"  content: {content}")
            if hasattr(msg, 'tool_calls') and msg.tool_calls:
                calls = []
                for tc in msg.tool_calls:
                    args_str = ", ".join(f"{k}={_format_arg(v)}" for k, v in tc['args'].items())
                    calls.append(f"{tc['name']}({args_str})")
                print(f"  tool_calls: {calls}")
            print()
            last_msg = msg
    return last_msg

In [ ]:
question = {'messages':[HumanMessage('오늘 날짜가 어떻게 되니?')]}

result = stream_agent(agent, question)

### 토큰 단위 스트리밍

stream_mode="updates"는 각 스텝이 끝난 시점의 결과를 한 번에 돌려줍니다.  
답변이 생성되는 과정을 토큰 단위로 확인하기 위해, stream_utils.py를 사용합니다.

- stream_print: 토큰과 도구 호출 기록을 print로 출력
- stream_with_markdown: 토큰을 누적해 마크다운으로 렌더링

In [ ]:
from stream_utils import stream_print, stream_with_markdown

In [ ]:
result = await stream_print(agent, '오늘 날짜를 확인하고, 올해가 며칠 남았는지 계산해줘.')

### 마크다운 렌더링 스트리밍

In [ ]:
result = await stream_with_markdown(agent, 'LangChain의 create_agent가 제공하는 기능을 검색해서 마크다운 표로 정리해줘.')

### 대화 메모리(Checkpointer)

지금까지의 에이전트는 invoke를 호출할 때마다 빈 메시지 목록에서 시작하므로, 직전 대화를 참조하지 못합니다.  
대화를 이어가려면 체크포인터를 연결해 매 스텝의 상태를 저장하도록 합니다.  

create_agent의 checkpointer 인자에 InMemorySaver를 전달하면 상태가 메모리에 보관됩니다.  
보관된 상태는 config의 thread_id로 구분하므로, 같은 thread_id로 호출한 대화끼리 메시지가 이어집니다.

In [ ]:
from langgraph.checkpoint.memory import InMemorySaver

memory_agent = create_agent(
    llm,
    tools=tools,
    system_prompt='주어진 도구를 사용하여 질문에 답변하세요. 앞선 대화에서 확인한 정보는 다시 묻지 말고 활용하세요.',
    checkpointer=InMemorySaver(),
)

config = {'configurable': {'thread_id': 'session-1'}}

In [ ]:
first = memory_agent.invoke(
    {'messages': [HumanMessage('내 이름은 김민준이고, 서울에서 데이터 분석 업무를 하고 있어.')]},
    config=config,
)

print(first['messages'][-1].text)

In [ ]:
second = memory_agent.invoke(
    {'messages': [HumanMessage('내 이름과 하는 일을 다시 알려줘.')]},
    config=config,
)

print(second['messages'][-1].text)

get_state로 특정 thread_id에 보관된 상태를 조회하면, 지금까지 오간 메시지 전체를 확인할 수 있습니다.

In [ ]:
state = memory_agent.get_state(config)

for message in state.values['messages']:
    print(f'{type(message).__name__}: {message.text}')

thread_id를 바꾸면 별도의 상태에서 대화가 시작되므로, 앞선 두 번의 대화는 참조되지 않습니다.

In [ ]:
new_config = {'configurable': {'thread_id': 'session-2'}}

third = memory_agent.invoke(
    {'messages': [HumanMessage('내 이름과 하는 일을 다시 알려줘.')]},
    config=new_config,
)

print(third['messages'][-1].text)

stream_with_markdown에 config를 함께 지정하면 astream_events로 그대로 전달되므로, 메모리를 연결한 에이전트에도 같은 방식으로 사용합니다.

In [ ]:
result = await stream_with_markdown(memory_agent, '내 이름으로 삼행시를 지어줘.', config=config)

### 정리

- stream_print, stream_with_markdown: astream_events로 토큰과 도구 실행을 실시간 출력하는 stream_utils.py의 헬퍼
- InMemorySaver: 대화 상태를 메모리에 보관하는 체크포인터
- thread_id: 하나의 대화 세션을 구분하는 키
- get_state: 특정 thread_id에 보관된 메시지 전체를 조회하는 메서드

InMemorySaver는 커널이 종료되면 보관하던 상태도 함께 사라집니다.  
대화를 파일이나 데이터베이스에 남기려면 SqliteSaver, PostgresSaver를 사용합니다.

## [실습] 유용한 툴 3개 추가하기

아래의 툴을 그대로 구성하거나 개선하고, LLM에 연결하여 실행해 봅시다.

In [ ]:
import os

BLOCKED_FILES = ['env', '.env', 'credentials', '.secret', 'id_rsa', '.pem']


def _resolve_within_base(file_path: str, base_dir: str | None = None) -> str | None:
    """file_path가 base_dir(기본: 현재 작업 디렉터리) 트리 안에 있으면
    정규화된 절대 경로를 반환, 벗어나면 None."""
    base = os.path.realpath(base_dir or os.getcwd())
    target = os.path.realpath(file_path)        # .. 및 심볼릭 링크까지 해소
    try:
        if os.path.commonpath([base, target]) == base:
            return target
    except ValueError:
        pass  # 서로 다른 드라이브(Windows) 등 비교 불가한 경우
    return None


def _is_blocked_name(file_path: str) -> bool:
    base = os.path.basename(file_path)
    return base in BLOCKED_FILES or any(file_path.endswith(b) for b in BLOCKED_FILES)


@tool
def read_file(file_path: str) -> str:
    """파일 내용을 읽습니다. file_path: 파일 경로"""
    if _is_blocked_name(file_path):
        return f"보안 정책에 의해 '{os.path.basename(file_path)}' 파일은 읽을 수 없습니다."
    safe = _resolve_within_base(file_path)
    if safe is None:
        return "보안 정책에 의해 현재 작업 디렉터리 밖의 파일은 읽을 수 없습니다."
    try:
        text = open(safe, encoding='utf-8').read()
        return text[:10000] + ('\n... (truncated)' if len(text) > 10000 else '')
    except Exception as e:
        return f'파일 읽기 오류: {e}'


@tool
def write_file(file_path: str, content: str) -> str:
    """파일에 내용을 작성. file_path: 경로, content: 작성할 내용"""
    if _is_blocked_name(file_path):
        return f"보안 정책에 의해 '{os.path.basename(file_path)}' 파일은 쓸 수 없습니다."
    safe = _resolve_within_base(file_path)        # makedirs 전에 검증
    if safe is None:
        return "보안 정책에 의해 현재 작업 디렉터리 밖에는 저장할 수 없습니다."
    try:
        d = os.path.dirname(safe)
        if d:
            os.makedirs(d, exist_ok=True)
        with open(safe, 'w', encoding='utf-8') as f:
            f.write(content)
        return f'파일 작성 완료: {safe} ({len(content)}자)'
    except Exception as e:
        return f'파일 작성 오류: {e}'

In [ ]:
from langchain_community.document_loaders import WebBaseLoader

@tool
def fetch_url(url: str) -> str:
    """URL의 웹페이지 내용을 가져옵니다. url: http(s):// 웹페이지 URL"""
    if not url.startswith(('http://', 'https://')):
        return f'오류: http(s):// URL만 지원합니다. 받은 값: {url}'
    try:
        docs = WebBaseLoader(url).load()
        content = docs[0].page_content if docs else '내용을 가져올 수 없습니다.'
        return content[:5000]
    except Exception as e:
        return f'fetch_url 오류: {type(e).__name__}: {e}'


Draw Image Tool: API 인증에 따라 실행 어려울 수 있음

In [ ]:
import base64
from zoneinfo import ZoneInfo
from datetime import datetime
from pathlib import Path
from IPython.display import Image

@tool
def draw_image(prompt:str) ->str:
    """
    텍스트 프롬프트를 기반으로 AI 이미지를 생성하고 로컬에 저장합니다.

    이 도구는 사용자가 이미지 생성, 그림 그리기, 시각적 콘텐츠 제작을
    요청할 때 사용합니다. 내부적으로 GPT 이미지 생성 모델을 호출합니다.

    Args:
        prompt (str): 생성할 이미지를 설명하는 텍스트.
            구체적이고 상세할수록 결과물 품질이 향상됩니다.

    Returns:
        str: 생성된 이미지의 저장 경로를 포함한 결과 메시지.
            성공 시 "✅ Image saved: outputs/generated_images/image_YYYYMMDD_HHMMSS_mmm.png" 형식.

    Note:
        - 이미지는 outputs/nano_banana2/ 디렉토리에 타임스탬프 기반 파일명으로 저장됨
        - 출력 형식: PNG
    """
    llm = ChatOpenAI(model="gpt-5.2", reasoning_effort='low')
    tool = {"type": "image_generation", "model": "gpt-image-2"}
    llm_with_tools = llm.bind_tools([tool])
    ai_message = llm_with_tools.invoke(prompt)

    image = next(item for item in ai_message.content_blocks if item["type"] == "image")

    kst = ZoneInfo("Asia/Seoul")
    ts = datetime.now(kst).strftime("%Y%m%d_%H%M%S_%f")[:-3]  # 밀리초까지
    out_dir = Path("outputs") / "generated_images"
    out_dir.mkdir(parents=True, exist_ok=True)

    out_path = out_dir / f"image_{ts}.png"
    out_path.write_bytes(base64.b64decode(image["base64"]))

    # (선택) 노트북에서 바로 보고 싶으면 주석 해제
    display(Image(filename=str(out_path)))

    return f"✅ Image saved: {out_path.as_posix()}"

# [실습] ReAct Agent에 새로운 도구 추가하기

아래 조건에 맞는 에이전트를 만들어보세요:

1. 새로운 도구를 1개 이상 추가
2. 에이전트에 적합한 시스템 프롬프트 작성
3. 여러 도구를 조합해야 하는 질문으로 테스트